In [1]:
from pyspark.sql import SparkSession

# 스파크 세션 생성
spark = SparkSession.builder \
    .appName("SparkTest") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.0


In [2]:
from pyspark.ml.feature import HashingTF, IDF, Tokenizer

# 1. DataFrame 생성
sentenceData = spark.createDataFrame(
    [(0, "Apache Spark is great"), (1, "Unstructured data analysis is powerful")],
    ["id", "sentence"],
)

# 2. 토큰화 (Tokenizer)
tokenizer = Tokenizer(inputCol="sentence", outputCol="words")
wordsData = tokenizer.transform(sentenceData)

# 3. TF(Term Frequency) 계산
hashingTF = HashingTF(
    inputCol="words", outputCol="rawFeatures", numFeatures=20
)
featurizedData = hashingTF.transform(wordsData)

# 4. IDF(Inverse Document Frequency) 계산
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)
rescaledData = idfModel.transform(featurizedData)

In [3]:
rescaledData.select("id", "sentence", "words", "rawFeatures", "features").show(
    truncate=False
)

+---+--------------------------------------+--------------------------------------------+----------------------------------------+-------------------------------------------------------------------------------------+
|id |sentence                              |words                                       |rawFeatures                             |features                                                                             |
+---+--------------------------------------+--------------------------------------------+----------------------------------------+-------------------------------------------------------------------------------------+
|0  |Apache Spark is great                 |[apache, spark, is, great]                  |(20,[3,6,9,10],[1.0,1.0,1.0,1.0])       |(20,[3,6,9,10],[0.4054651081081644,0.0,0.0,0.4054651081081644])                      |
|1  |Unstructured data analysis is powerful|[unstructured, data, analysis, is, powerful]|(20,[0,6,9,15,16],[1.0,1.0,1.0,1.0,1.0])|(2

In [4]:
wordsData.show(truncate=False)

+---+--------------------------------------+--------------------------------------------+
|id |sentence                              |words                                       |
+---+--------------------------------------+--------------------------------------------+
|0  |Apache Spark is great                 |[apache, spark, is, great]                  |
|1  |Unstructured data analysis is powerful|[unstructured, data, analysis, is, powerful]|
+---+--------------------------------------+--------------------------------------------+



In [5]:
featurizedData.show(truncate=False)

+---+--------------------------------------+--------------------------------------------+----------------------------------------+
|id |sentence                              |words                                       |rawFeatures                             |
+---+--------------------------------------+--------------------------------------------+----------------------------------------+
|0  |Apache Spark is great                 |[apache, spark, is, great]                  |(20,[3,6,9,10],[1.0,1.0,1.0,1.0])       |
|1  |Unstructured data analysis is powerful|[unstructured, data, analysis, is, powerful]|(20,[0,6,9,15,16],[1.0,1.0,1.0,1.0,1.0])|
+---+--------------------------------------+--------------------------------------------+----------------------------------------+



In [6]:
rescaledData.select("id", "features").show(truncate=False)

+---+-------------------------------------------------------------------------------------+
|id |features                                                                             |
+---+-------------------------------------------------------------------------------------+
|0  |(20,[3,6,9,10],[0.4054651081081644,0.0,0.0,0.4054651081081644])                      |
|1  |(20,[0,6,9,15,16],[0.4054651081081644,0.0,0.0,0.4054651081081644,0.4054651081081644])|
+---+-------------------------------------------------------------------------------------+



In [7]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Spark DataFrame에서 features 컬럼 데이터를 Pandas로 수집
# (데이터가 적으므로 toPandas()를 사용해도 무방합니다)
features_list = [row.features for row in rescaledData.select("features").collect()]

# 2. Spark SparseVector를 일반 numpy 배열로 변환
# (scikit-learn이 이해할 수 있는 형태)
dense_features = np.array([v.toArray() for v in features_list])

# 3. 코사인 유사도 계산
similarity_matrix = cosine_similarity(dense_features)

print("--- 문서 간 코사인 유사도 매트릭스 ---")
print(similarity_matrix)

# 4. 첫 번째 문장과 두 번째 문장의 유사도 값만 예쁘게 출력
sim_score = similarity_matrix[0][1]

print(
    f"\n문장 0 ('Apache Spark is great')과 문장 1 ('Unstructured data analysis is powerful')의 유사도: {sim_score:.4f}"
)

--- 문서 간 코사인 유사도 매트릭스 ---
[[1. 0.]
 [0. 1.]]

문장 0 ('Apache Spark is great')과 문장 1 ('Unstructured data analysis is powerful')의 유사도: 0.0000


In [8]:
# 한글 적용 여부 확인

from pyspark.ml.feature import HashingTF, IDF, Tokenizer

# 1. DataFrame 생성
sentenceData = spark.createDataFrame(
    [(0, "아이브 장원영은 이쁘다"), (1, "이번 아이브 해외투어는 다소 아쉬웠다") , (2 , '이번에 이서는 특이한 의상을 입었다')],
    ["id", "sentence"],
)

# 2. 토큰화 (Tokenizer)
tokenizer = Tokenizer(inputCol="sentence", outputCol="words")
wordsData = tokenizer.transform(sentenceData)

# 3. TF(Term Frequency) 계산
hashingTF = HashingTF(
    inputCol="words", outputCol="rawFeatures", numFeatures=20
)
featurizedData = hashingTF.transform(wordsData)

# 4. IDF(Inverse Document Frequency) 계산
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)
rescaledData = idfModel.transform(featurizedData)

In [9]:
rescaledData.select("id", "sentence", "words", "rawFeatures", "features").show(
    truncate=False
)

+---+------------------------------------+------------------------------------------+-----------------------------------+------------------------------------------------------------------------------------------------+
|id |sentence                            |words                                     |rawFeatures                        |features                                                                                        |
+---+------------------------------------+------------------------------------------+-----------------------------------+------------------------------------------------------------------------------------------------+
|0  |아이브 장원영은 이쁘다              |[아이브, 장원영은, 이쁘다]                |(20,[0,2,9],[1.0,1.0,1.0])         |(20,[0,2,9],[0.28768207245178085,0.28768207245178085,0.28768207245178085])                      |
|1  |이번 아이브 해외투어는 다소 아쉬웠다|[이번, 아이브, 해외투어는, 다소, 아쉬웠다]|(20,[2,4,5,9],[1.0,1.0,1.0,2.0])   |(20,[2,4,5,9],[0.28768207245178085,0.6931471805599453,0

In [10]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Spark DataFrame에서 features 컬럼 데이터를 Pandas로 수집
# (데이터가 적으므로 toPandas()를 사용해도 무방합니다)
features_list = [row.features for row in rescaledData.select("features").collect()]

# 2. Spark SparseVector를 일반 numpy 배열로 변환
# (scikit-learn이 이해할 수 있는 형태)
dense_features = np.array([v.toArray() for v in features_list])

# 3. 코사인 유사도 계산
similarity_matrix = cosine_similarity(dense_features)

print("--- 문서 간 코사인 유사도 매트릭스 ---")
print(similarity_matrix)

# 4. 첫 번째 문장과 두 번째 문장의 유사도 값만 예쁘게 출력
sim_score = similarity_matrix[0][1]

print(
    f"\n문장 0 ('아이브 장원영은 이쁘다')과 문장 1 ('이번 아이브 해외투어는 다소 아쉬웠다')의 유사도: {sim_score:.4f}"
)

# 5. 첫 번째 문장과 세 번째 문장의 유사도 값만 예쁘게 출력
sim_score = similarity_matrix[0][2]

print(
    f"\n문장 0 ('아이브 장원영은 이쁘다')과 문장 2 ('이번에 이서는 특이한 의상을 입었다')의 유사도: {sim_score:.4f}"
)

# 6. 두 번째 문장과 세 번째 문장의 유사도 값만 예쁘게 출력
sim_score = similarity_matrix[1][2]

print(
    f"\n문장 1 ('이번 아이브 해외투어는 다소 아쉬웠다')과 문장 2 ('이번에 이서는 특이한 의상을 입었다')의 유사도: {sim_score:.4f}"
)

--- 문서 간 코사인 유사도 매트릭스 ---
[[1.         0.42497926 0.09645056]
 [0.42497926 1.         0.        ]
 [0.09645056 0.         1.        ]]

문장 0 ('아이브 장원영은 이쁘다')과 문장 1 ('이번 아이브 해외투어는 다소 아쉬웠다')의 유사도: 0.4250

문장 0 ('아이브 장원영은 이쁘다')과 문장 2 ('이번에 이서는 특이한 의상을 입었다')의 유사도: 0.0965

문장 1 ('이번 아이브 해외투어는 다소 아쉬웠다')과 문장 2 ('이번에 이서는 특이한 의상을 입었다')의 유사도: 0.0000


In [11]:
from pyspark.ml.feature import CountVectorizer, IDF, Tokenizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 데이터프레임 생성
sentenceData = spark.createDataFrame(
    [
        (0, "아이브 장원영은 이쁘다"),
        (1, "이번 아이브 해외투어는 다소 아쉬웠다"),
        (2, "이번에 이서는 특이한 의상을 입었다"),
    ],
    ["id", "sentence"],
)

# 2. 토큰화 (주의: 한글은 형태소 분석기가 아니면 띄어쓰기 기준이므로 음절/어절 단위가 됩니다)
tokenizer = Tokenizer(inputCol="sentence", outputCol="words")
wordsData = tokenizer.transform(sentenceData)

# 3. HashingTF 대신 CountVectorizer 사용 (해시 충돌 방지)
cv = CountVectorizer(inputCol="words", outputCol="rawFeatures")
cvModel = cv.fit(wordsData)
featurizedData = cvModel.transform(wordsData)

# 4. IDF 계산
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)
rescaledData = idfModel.transform(featurizedData)

# 5. 유사도 확인
features_list = [row.features for row in rescaledData.select("features").collect()]
dense_features = np.array([v.toArray() for v in features_list])
print(cosine_similarity(dense_features))

[[1.         0.05721813 0.        ]
 [0.05721813 1.         0.        ]
 [0.         0.         1.        ]]


In [12]:
# 이번엔 형태소 분석까지 진행

# 1. 시스템 패키지 매니저를 통해 자바(Java) 설치 (KoNLPy 구동용)
!apt-get update && apt-get install -y openjdk-11-jdk

# 2. 파이썬 형태소 분석 라이브러리 설치
!pip install jpype1 konlpy

Reading package lists... Done
E: List directory /var/lib/apt/lists/partial is missing. - Acquire (13: Permission denied)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.4/439.4 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 57.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 68.0 MB/s eta 0:00:0000:0100:01


In [13]:
from konlpy.tag import Okt
from pyspark.ml.feature import CountVectorizer, IDF
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. KoNLPy Okt 객체 생성
okt = Okt()

# 2. 원본 데이터 준비
raw_data = [
    (0, "아이브 장원영은 이쁘다"),
    (1, "이번 아이브 해외투어는 다소 아쉬웠다"),
    (2, "이번에 이서는 특이한 의상을 입었다"),
]

# 3. KoNLPy를 이용해 문장을 형태소(여기서는 명사 중심 또는 형태소 리스트)로 미리 분해
# 예: "아이브 장원영은 이쁘다" -> ['아이브', '장원영', '이쁘다']
processed_data = []
for row_id, sentence in raw_data:
  # 묵음/조사를 제외하고 어절 또는 형태소 추출 (여기서는 morphs 또는 nouns 사용)
  # 명사만 추출하고 싶다면 okt.nouns(sentence)를 사용해도 좋습니다.
  tokens = okt.morphs(sentence)
  processed_data.append((row_id, sentence, tokens))

# 4. 파이썬 리스트를 PySpark DataFrame으로 변환
# (schema를 명시하여 words 컬럼을 배열 형태로 생성)
from pyspark.sql.types import ArrayType, IntegerType, StringType, StructField, StructType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("sentence", StringType(), True),
    StructField("words", ArrayType(StringType()), True),
])

wordsData = spark.createDataFrame(processed_data, schema=schema)

# 5. CountVectorizer 적용 (해시 충돌 방지)
cv = CountVectorizer(inputCol="words", outputCol="rawFeatures")
cvModel = cv.fit(wordsData)
featurizedData = cvModel.transform(wordsData)

# 6. IDF 계산
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)
rescaledData = idfModel.transform(featurizedData)

# 7. scikit-learn을 이용한 코사인 유사도 계산
features_list = [row.features for row in rescaledData.select("features").collect()]
dense_features = np.array([v.toArray() for v in features_list])

print("--- KoNLPy 형태소 분석 적용 후 유사도 매트릭스 ---")
print(cosine_similarity(dense_features))

# 6. 두 번째 문장과 세 번째 문장의 유사도 값만 예쁘게 출력
sim_score = cosine_similarity(dense_features)[1][2]

print(
    f"\n문장 1 ('이번 아이브 해외투어는 다소 아쉬웠다')과 문장 2 ('이번에 이서는 특이한 의상을 입었다')의 유사도: {sim_score:.4f}"
)

--- KoNLPy 형태소 분석 적용 후 유사도 매트릭스 ---
[[1.         0.04550684 0.        ]
 [0.04550684 1.         0.06435638]
 [0.         0.06435638 1.        ]]

문장 1 ('이번 아이브 해외투어는 다소 아쉬웠다')과 문장 2 ('이번에 이서는 특이한 의상을 입었다')의 유사도: 0.0644


In [14]:
from konlpy.tag import Okt
from pyspark.ml.feature import CountVectorizer, IDF
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. KoNLPy Okt 객체 생성
okt = Okt()

# 2. 원본 데이터 준비
raw_data = [
    (0, "아이브 장원영은 이쁘다"),
    (1, "이번 아이브 해외투어는 다소 아쉬웠다"),
    (2, "이번에 이서는 특이한 의상을 입었다"),
]

# 3. KoNLPy를 이용해 문장을 형태소(여기서는 명사 중심 또는 형태소 리스트)로 미리 분해
# 예: "아이브 장원영은 이쁘다" -> ['아이브', '장원영', '이쁘다']
processed_data = []
for row_id, sentence in raw_data:
  # 묵음/조사를 제외하고 어절 또는 형태소 추출 (여기서는 morphs 또는 nouns 사용)
  # 명사만 추출하고 싶다면 okt.nouns(sentence)를 사용해도 좋습니다.
  # morphs 대신 nouns를 쓰면 명사만 쏙쏙 골라냅니다.
  tokens = okt.nouns(sentence)
  processed_data.append((row_id, sentence, tokens))

# 4. 파이썬 리스트를 PySpark DataFrame으로 변환
# (schema를 명시하여 words 컬럼을 배열 형태로 생성)
from pyspark.sql.types import ArrayType, IntegerType, StringType, StructField, StructType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("sentence", StringType(), True),
    StructField("words", ArrayType(StringType()), True),
])

wordsData = spark.createDataFrame(processed_data, schema=schema)

# 5. CountVectorizer 적용 (해시 충돌 방지)
cv = CountVectorizer(inputCol="words", outputCol="rawFeatures")
cvModel = cv.fit(wordsData)
featurizedData = cvModel.transform(wordsData)

# 6. IDF 계산
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)
rescaledData = idfModel.transform(featurizedData)

# 7. scikit-learn을 이용한 코사인 유사도 계산
features_list = [row.features for row in rescaledData.select("features").collect()]
dense_features = np.array([v.toArray() for v in features_list])

print("--- KoNLPy 형태소 분석 적용 후 유사도 매트릭스 ---")
print(cosine_similarity(dense_features))

# 6. 두 번째 문장과 세 번째 문장의 유사도 값만 예쁘게 출력
sim_score = cosine_similarity(dense_features)[1][2]

print(
    f"\n문장 1 ('이번 아이브 해외투어는 다소 아쉬웠다')과 문장 2 ('이번에 이서는 특이한 의상을 입었다')의 유사도: {sim_score:.4f}"
)

--- KoNLPy 형태소 분석 적용 후 유사도 매트릭스 ---
[[1.         0.08699555 0.        ]
 [0.08699555 1.         0.06390764]
 [0.         0.06390764 1.        ]]

문장 1 ('이번 아이브 해외투어는 다소 아쉬웠다')과 문장 2 ('이번에 이서는 특이한 의상을 입었다')의 유사도: 0.0639


In [15]:
# 단어 빈도수(TF) 상위 랭킹 뽑기

import pandas as pd

# 1. CountVectorizer가 학습한 전체 단어 사전(Vocabulary) 가져오기
vocab = cvModel.vocabulary

# 2. 모든 문서의 rawFeatures(희소 벡터)를 더해서 전체 단어별 총 빈도수 계산
from pyspark.sql.functions import col, sum as spark_sum

# 각 행의 rawFeatures 배열을 모아서 총합 계산
# (Spark 에서는 VectorAssembler나 간단히 Pandas로 변환하여 처리하는 것이 직관적입니다)
all_raw_features = [row.rawFeatures.toArray() for row in featurizedData.select("rawFeatures").collect()]
total_term_frequencies = sum(all_raw_features)

# 3. 단어와 빈도수를 묶어서 데이터프레임으로 변환
tf_df = pd.DataFrame({
    'word': vocab,
    'total_tf': total_term_frequencies
}).sort_values(by='total_tf', ascending=False).reset_index(drop=True)

print("--- 🏆 전체 단어 빈도수 (TF) 상위 랭킹 ---")
print(tf_df.head(10))

--- 🏆 전체 단어 빈도수 (TF) 상위 랭킹 ---
  word  total_tf
0  아이브       2.0
1   이번       2.0
2   의상       1.0
3   다소       1.0
4   해외       1.0
5   이서       1.0
6  장원영       1.0
7   투어       1.0


In [16]:
# rescaledData에서 id, sentence, words, features 컬럼을 가져와서 분석
results = rescaledData.select("id", "sentence", "words", "features").collect()

print("--- 🔍 문서별 TF-IDF 가중치 상위 핵심 단어 ---")
for row in results:
    row_id = row.id
    sentence = row.sentence
    words = row.words
    features = row.features # SparseVector (indices, values)
    
    # 인덱스와 IDF 가중치 맵핑
    indices = features.indices
    values = features.values
    
    # (단어, 가중치) 형태의 리스트 생성
    word_weights = []
    for idx, val in zip(indices, values):
        word = vocab[idx] # 인덱스에 해당하는 실제 단어 이름
        word_weights.append((word, val))
        
    # 가중치(TF-IDF)가 높은 순서대로 정렬
    word_weights.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n[문장 {row_id}] {sentence}")
    print(f"ㄴ 토큰 목록: {words}")
    print(f"ㄴ 핵심 키워드 및 가중치(TF-IDF): {word_weights}")

--- 🔍 문서별 TF-IDF 가중치 상위 핵심 단어 ---

[문장 0] 아이브 장원영은 이쁘다
ㄴ 토큰 목록: ['아이브', '장원영']
ㄴ 핵심 키워드 및 가중치(TF-IDF): [('장원영', 0.6931471805599453), ('아이브', 0.28768207245178085)]

[문장 1] 이번 아이브 해외투어는 다소 아쉬웠다
ㄴ 토큰 목록: ['이번', '아이브', '해외', '투어', '다소']
ㄴ 핵심 키워드 및 가중치(TF-IDF): [('다소', 0.6931471805599453), ('해외', 0.6931471805599453), ('투어', 0.6931471805599453), ('아이브', 0.28768207245178085), ('이번', 0.28768207245178085)]

[문장 2] 이번에 이서는 특이한 의상을 입었다
ㄴ 토큰 목록: ['이번', '이서', '의상']
ㄴ 핵심 키워드 및 가중치(TF-IDF): [('의상', 0.6931471805599453), ('이서', 0.6931471805599453), ('이번', 0.28768207245178085)]


In [17]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF, NGram, Tokenizer

# 1. 파이프라인 단계 정의 (토큰화 -> N-gram 추가 -> TF -> IDF)
tokenizer = Tokenizer(inputCol="sentence", outputCol="words")
ngram = NGram(n=2, inputCol="words", outputCol="ngrams")  # 2개 단어 묶음 추출
cv = CountVectorizer(inputCol="ngrams", outputCol="rawFeatures")
idf = IDF(inputCol="rawFeatures", outputCol="features")

# 2. 파이프라인 객체로 묶기
pipeline = Pipeline(stages=[tokenizer, ngram, cv, idf])

# 3. 데이터프레임에 한 번에 학습 및 변환 적용
model = pipeline.fit(sentenceData)
pipeline_result = model.transform(sentenceData)

pipeline_result.select("sentence", "ngrams", "features").show(
    truncate=False
)

+------------------------------------+----------------------------------------------------------------+--------------------------------------------------------------------------------------------+
|sentence                            |ngrams                                                          |features                                                                                    |
+------------------------------------+----------------------------------------------------------------+--------------------------------------------------------------------------------------------+
|아이브 장원영은 이쁘다              |[아이브 장원영은, 장원영은 이쁘다]                              |(10,[5,7],[0.6931471805599453,0.6931471805599453])                                          |
|이번 아이브 해외투어는 다소 아쉬웠다|[이번 아이브, 아이브 해외투어는, 해외투어는 다소, 다소 아쉬웠다]|(10,[2,3,6,8],[0.6931471805599453,0.6931471805599453,0.6931471805599453,0.6931471805599453])|
|이번에 이서는 특이한 의상을 입었다  |[이번에 이서는, 이서는 특이한, 특이한 의상을, 의상을 입었다]    |(10,[0,1,4,9],[0.

In [18]:
# PySpark N-gram 및 유사도 수치화 코드

from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF, NGram, Tokenizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 샘플 데이터프레임 생성
sentenceData = spark.createDataFrame(
    [
        (0, "아이브 장원영은 이쁘다"),
        (1, "이번 아이브 해외투어는 다소 아쉬웠다"),
        (2, "이번에 이서는 특이한 의상을 입었다"),
    ],
    ["id", "sentence"],
)

# 2. Spark ML Pipeline 구성
# Step 1: 토큰화 (띄어쓰기 기준)
tokenizer = Tokenizer(inputCol="sentence", outputCol="words")

# Step 2: N-gram 적용 (n=2 이면 2개의 연속된 단어를 하나로 묶음. 예: "아이브 해외투어")
ngram = NGram(n=2, inputCol="words", outputCol="ngrams")

# Step 3: CountVectorizer (해시 충돌 방지 및 빈도 계산)
cv = CountVectorizer(inputCol="ngrams", outputCol="rawFeatures")

# Step 4: IDF (역문서 빈도 가중치 부여)
idf = IDF(inputCol="rawFeatures", outputCol="features")

# 3. 파이프라인으로 연결
pipeline = Pipeline(stages=[tokenizer, ngram, cv, idf])

# 4. 모델 학습 및 데이터 변환
model = pipeline.fit(sentenceData)
rescaledData = model.transform(sentenceData)

# 5. 결과 확인 (N-gram 결과가 어떻게 나왔는지 체크)
print("--- 📝 N-gram 적용 결과 확인 ---")
rescaledData.select("id", "sentence", "ngrams", "features").show(
    truncate=False
)

# 6. 코사인 유사도 수치화 계산
features_list = [row.features for row in rescaledData.select("features").collect()]
dense_features = np.array([v.toArray() for v in features_list])
similarity_matrix = cosine_similarity(dense_features)

print("--- 📊 N-gram 기반 문서 간 코사인 유사도 매트릭스 ---")
print(similarity_matrix)

--- 📝 N-gram 적용 결과 확인 ---
+---+------------------------------------+----------------------------------------------------------------+--------------------------------------------------------------------------------------------+
|id |sentence                            |ngrams                                                          |features                                                                                    |
+---+------------------------------------+----------------------------------------------------------------+--------------------------------------------------------------------------------------------+
|0  |아이브 장원영은 이쁘다              |[아이브 장원영은, 장원영은 이쁘다]                              |(10,[5,7],[0.6931471805599453,0.6931471805599453])                                          |
|1  |이번 아이브 해외투어는 다소 아쉬웠다|[이번 아이브, 아이브 해외투어는, 해외투어는 다소, 다소 아쉬웠다]|(10,[2,3,6,8],[0.6931471805599453,0.6931471805599453,0.6931471805599453,0.6931471805599453])|
|2  |이번에 이서는 특이한 의상을 입었다  |[이번에 이서는

In [19]:
# N-gram은 문장들이 길고 "동일한 표현이나 구문(예: '해외 투어', '아이브 멤버')"을 공유할 때 
# 진가가 발휘되기 때문에 문장 샘플을 교체

sentenceData = spark.createDataFrame(
    [
        (0, "아이브 해외투어는 성공적으로 끝난거 같다"),
        (1, "이번 아이브 해외투어는 다소 아쉬웠다"),
        (2, "이번에 이서는 특이한 의상을 입었다"),
        (3, "이서는 특이한 노래도 냈었다")
    ],
    ["id", "sentence"],
)

# Spark ML Pipeline 구성
# Step 1: 토큰화 (띄어쓰기 기준)
tokenizer = Tokenizer(inputCol="sentence", outputCol="words")

# Step 2: N-gram 적용 (n=2 이면 2개의 연속된 단어를 하나로 묶음. 예: "아이브 해외투어")
ngram = NGram(n=2, inputCol="words", outputCol="ngrams")

# Step 3: CountVectorizer (해시 충돌 방지 및 빈도 계산)
cv = CountVectorizer(inputCol="ngrams", outputCol="rawFeatures")

# Step 4: IDF (역문서 빈도 가중치 부여)
idf = IDF(inputCol="rawFeatures", outputCol="features")

# 파이프라인으로 연결
pipeline = Pipeline(stages=[tokenizer, ngram, cv, idf])

# 모델 학습 및 데이터 변환
model = pipeline.fit(sentenceData)
rescaledData = model.transform(sentenceData)

# 결과 확인 (N-gram 결과가 어떻게 나왔는지 체크)
print("--- 📝 N-gram 적용 결과 확인 ---")
rescaledData.select("id", "sentence", "ngrams", "features").show(
    truncate=False
)

# 코사인 유사도 수치화 계산
features_list = [row.features for row in rescaledData.select("features").collect()]
dense_features = np.array([v.toArray() for v in features_list])
similarity_matrix = cosine_similarity(dense_features)

print("--- 📊 N-gram 기반 문서 간 코사인 유사도 매트릭스 ---")
print(similarity_matrix)

# 0 / 1 번째 문장이 0.09
# 2 / 3 번째 문장이 0.11 유사도

--- 📝 N-gram 적용 결과 확인 ---
+---+----------------------------------------+--------------------------------------------------------------------------+---------------------------------------------------------------------------------------------+
|id |sentence                                |ngrams                                                                    |features                                                                                     |
+---+----------------------------------------+--------------------------------------------------------------------------+---------------------------------------------------------------------------------------------+
|0  |아이브 해외투어는 성공적으로 끝난거 같다|[아이브 해외투어는, 해외투어는 성공적으로, 성공적으로 끝난거, 끝난거 같다]|(13,[1,4,9,11],[0.5108256237659907,0.9162907318741551,0.9162907318741551,0.9162907318741551])|
|1  |이번 아이브 해외투어는 다소 아쉬웠다    |[이번 아이브, 아이브 해외투어는, 해외투어는 다소, 다소 아쉬웠다]          |(13,[1,5,6,10],[0.5108256237659907,0.9162907318741551,0.9162907318741551,0.91629

In [20]:
# PySpark ML에는 기본적으로 영어나 주요 언어의 불용어 사전(StopWordsRemover)이 내장되어 있지만, 
# 한국어는 조사이거나 의미 없는 어미, 부사 등이 많기 때문에 
# 커스텀 불용어 리스트를 만들어 제거하는 것이 가장 확실

# 불용어 정제 및 N-gram 유사도 분석 코드

from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF, NGram, StopWordsRemover, Tokenizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 제시해주신 데이터프레임 생성
sentenceData = spark.createDataFrame(
    [
        (0, "아이브 해외투어는 성공적으로 끝난거 같다"),
        (1, "이번 아이브 해외투어는 다소 아쉬웠다"),
        (2, "이번에 이서는 특이한 의상을 입었다"),
        (3, "이서는 특이한 노래도 냈었다")
    ],
    ["id", "sentence"],
)

# 2. 한국어 커스텀 불용어(Stopwords) 정의
# (조사, 의미 없는 부사, 서술어 어미 등 문맥 파악에 방해되는 단어들)
korean_stopwords = [
    "은", "는", "이", "가", "을", "를", "의", "에", "와", "과", "도", 
    "는", "다", "거", "것", "같다", "였다", "었다", "았다", "에서", "로", "으로"
]

# 3. Spark ML Pipeline 구성
# Step 1: 토큰화 (띄어쓰기 기준)
tokenizer = Tokenizer(inputCol="sentence", outputCol="raw_words")

# Step 2: 불용어 제거 (StopWordsRemover)
# 띄어쓰기로 쪼개진 단어 중 불용어 리스트에 포함된 단어들을 필터링합니다.
stopwords_remover = StopWordsRemover(
    inputCol="raw_words", 
    outputCol="words", 
    stopWords=korean_stopwords
)

# Step 3: N-gram 적용 (n=2, 불용어가 제거된 깨끗한 단어들끼리 묶음)
ngram = NGram(n=2, inputCol="words", outputCol="ngrams")

# Step 4: CountVectorizer & IDF
cv = CountVectorizer(inputCol="ngrams", outputCol="rawFeatures")
idf = IDF(inputCol="rawFeatures", outputCol="features")

# 4. 파이프라인 결합 및 실행
pipeline = Pipeline(stages=[tokenizer, stopwords_remover, ngram, cv, idf])
model = pipeline.fit(sentenceData)
rescaledData = model.transform(sentenceData)

# 5. 불용어 제거 및 N-gram 결과 확인
print("--- 📝 불용어 제거 및 N-gram 변환 결과 ---")
rescaledData.select("id", "sentence", "words", "ngrams").show(truncate=False)

# 6. 코사인 유사도 수치화 매트릭스 계산
features_list = [row.features for row in rescaledData.select("features").collect()]
dense_features = np.array([v.toArray() for v in features_list])
similarity_matrix = cosine_similarity(dense_features)

print("--- 📊 불용어 정제 후 N-gram 기반 코사인 유사도 매트릭스 ---")
print(similarity_matrix)

# 0 / 1 번째 문장이 0.11
# 2 / 3 번째 문장이 0.11 유사도

--- 📝 불용어 제거 및 N-gram 변환 결과 ---
+---+----------------------------------------+------------------------------------------+----------------------------------------------------------------+
|id |sentence                                |words                                     |ngrams                                                          |
+---+----------------------------------------+------------------------------------------+----------------------------------------------------------------+
|0  |아이브 해외투어는 성공적으로 끝난거 같다|[아이브, 해외투어는, 성공적으로, 끝난거]  |[아이브 해외투어는, 해외투어는 성공적으로, 성공적으로 끝난거]   |
|1  |이번 아이브 해외투어는 다소 아쉬웠다    |[이번, 아이브, 해외투어는, 다소, 아쉬웠다]|[이번 아이브, 아이브 해외투어는, 해외투어는 다소, 다소 아쉬웠다]|
|2  |이번에 이서는 특이한 의상을 입었다      |[이번에, 이서는, 특이한, 의상을, 입었다]  |[이번에 이서는, 이서는 특이한, 특이한 의상을, 의상을 입었다]    |
|3  |이서는 특이한 노래도 냈었다             |[이서는, 특이한, 노래도, 냈었다]          |[이서는 특이한, 특이한 노래도, 노래도 냈었다]                   |
+---+----------------------------------------+------------------------------------------+--------

In [21]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF, NGram, StopWordsRemover, Tokenizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 제시해주신 데이터프레임 생성
sentenceData = spark.createDataFrame(
    [
        (0, "아이브 해외투어는 성공적으로 끝난거 같다"),
        (1, "이번 아이브 해외투어는 다소 아쉬웠다"),
        (2, "이번에 이서는 특이한 의상을 입었다"),
        (3, "이서도 특이한 노래도 냈었다")
    ],
    ["id", "sentence"],
)

# 2. 한국어 커스텀 불용어(Stopwords) 정의
# (조사, 의미 없는 부사, 서술어 어미 등 문맥 파악에 방해되는 단어들)
korean_stopwords = [
    "은", "는", "이", "가", "을", "를", "의", "에", "와", "과", "도", 
    "는", "다", "거", "것", "같다", "였다", "었다", "았다", "에서", "로", "으로"
]

# 3. Spark ML Pipeline 구성
# Step 1: 토큰화 (띄어쓰기 기준)
tokenizer = Tokenizer(inputCol="sentence", outputCol="raw_words")

# Step 2: 불용어 제거 (StopWordsRemover)
# 띄어쓰기로 쪼개진 단어 중 불용어 리스트에 포함된 단어들을 필터링합니다.
stopwords_remover = StopWordsRemover(
    inputCol="raw_words", 
    outputCol="words", 
    stopWords=korean_stopwords
)

# Step 3: N-gram 적용 (n=2, 불용어가 제거된 깨끗한 단어들끼리 묶음)
ngram = NGram(n=2, inputCol="words", outputCol="ngrams")

# Step 4: CountVectorizer & IDF
cv = CountVectorizer(inputCol="ngrams", outputCol="rawFeatures")
idf = IDF(inputCol="rawFeatures", outputCol="features")

# 4. 파이프라인 결합 및 실행
pipeline = Pipeline(stages=[tokenizer, stopwords_remover, ngram, cv, idf])
model = pipeline.fit(sentenceData)
rescaledData = model.transform(sentenceData)

# 5. 불용어 제거 및 N-gram 결과 확인
print("--- 📝 불용어 제거 및 N-gram 변환 결과 ---")
rescaledData.select("id", "sentence", "words", "ngrams").show(truncate=False)

# 6. 코사인 유사도 수치화 매트릭스 계산
features_list = [row.features for row in rescaledData.select("features").collect()]
dense_features = np.array([v.toArray() for v in features_list])
similarity_matrix = cosine_similarity(dense_features)

print("--- 📊 불용어 정제 후 N-gram 기반 코사인 유사도 매트릭스 ---")
print(similarity_matrix)

# 0 / 1 번째 문장이 0.11
# 2 / 3 번째 문장이 0 유사도

--- 📝 불용어 제거 및 N-gram 변환 결과 ---
+---+----------------------------------------+------------------------------------------+----------------------------------------------------------------+
|id |sentence                                |words                                     |ngrams                                                          |
+---+----------------------------------------+------------------------------------------+----------------------------------------------------------------+
|0  |아이브 해외투어는 성공적으로 끝난거 같다|[아이브, 해외투어는, 성공적으로, 끝난거]  |[아이브 해외투어는, 해외투어는 성공적으로, 성공적으로 끝난거]   |
|1  |이번 아이브 해외투어는 다소 아쉬웠다    |[이번, 아이브, 해외투어는, 다소, 아쉬웠다]|[이번 아이브, 아이브 해외투어는, 해외투어는 다소, 다소 아쉬웠다]|
|2  |이번에 이서는 특이한 의상을 입었다      |[이번에, 이서는, 특이한, 의상을, 입었다]  |[이번에 이서는, 이서는 특이한, 특이한 의상을, 의상을 입었다]    |
|3  |이서도 특이한 노래도 냈었다             |[이서도, 특이한, 노래도, 냈었다]          |[이서도 특이한, 특이한 노래도, 노래도 냈었다]                   |
+---+----------------------------------------+------------------------------------------+--------

In [22]:
# 2 / 3 번째 문장이 0 유사도

# 왜 유사도가 0이 나왔을까요?

# 1. 띄어쓰기 기준 토큰화의 한계

# 현재 사용 중인 Tokenizer는 띄어쓰기 단위로만 단어를 쪼갭니다.

# 문장 2 의 주어 부분: "이번에" (불용어 제거 후 ['이번'] 또는 ['이번에']) + "이서는" -->> 토큰은 "이서는"로 생성됩니다.
# 문장 3 의 주어 부분: "이서도" -->> 토큰은 "이서도"로 생성됩니다

# 2. 불용어 사전에 "는", "도"만 있고 "서"나 통째로 된 단어가 걸러지지 않음

# 단어 자체에 붙어 있는 어미나 조사가 깔끔하게 떨어지지 않고 "이서는"와 "이서도"라는 서로 완전히 다른 어절 문자열로 인식

# 그 결과 N-gram (n = 2) 을 묶었을 때도 두 문장이 공유하는 구절이 완전히 어긋나 버려 코사인 유사도가 0

# 띄어쓰기 기반 Tokenizer 대신, 앞서 다뤘던 KoNLPy(Okt 등)를 사용해 단어의 원형("이서")을 추출한 뒤 불용어를 제거해야 
# "이서는"와 "이서도"가 모두 "이서"라는 공통 키워드로 정확히 매칭

from konlpy.tag import Okt
from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF, NGram
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 원본 데이터 준비
raw_data = [
    (0, "아이브 해외투어는 성공적으로 끝난거 같다"),
    (1, "이번 아이브 해외투어는 다소 아쉬웠다"),
    (2, "이번에 이서는 특이한 의상을 입었다"),
    (3, "이서도 특이한 노래도 냈었다")
]

# 2. 파이썬 단(Driver)에서 KoNLPy로 안전하게 토큰화 및 불용어 제거 수행
okt = Okt()

processed_rows = []
for row_id, sentence in raw_data:
    # 형태소 분석 및 품사 태깅 후, 조사/어미/특수문자 제외하고 단어만 추출
    pos_results = okt.pos(sentence)
    tokens = [word for word, pos in pos_results if pos not in ['Josa', 'Eomi', 'Punctuation']]
    processed_rows.append((row_id, sentence, tokens))

# 3. 전처리가 완료된 데이터를 Spark DataFrame으로 생성
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("sentence", StringType(), True),
    StructField("words", ArrayType(StringType()), True)
])

sentenceData = spark.createDataFrame(processed_rows, schema=schema)

# 4. Spark ML Pipeline 구성 (토큰화 UDF는 이미 거쳤으므로 N-gram부터 시작)
ngram = NGram(n=2, inputCol="words", outputCol="ngrams")
cv = CountVectorizer(inputCol="ngrams", outputCol="rawFeatures")
idf = IDF(inputCol="rawFeatures", outputCol="features")

pipeline = Pipeline(stages=[ngram, cv, idf])
model = pipeline.fit(sentenceData)
rescaledData = model.transform(sentenceData)

# 5. 결과 확인
print("--- 📝 토큰화 및 N-gram 결과 ---")
rescaledData.select("id", "sentence", "words", "ngrams").show(truncate=False)

# 6. 코사인 유사도 수치화 계산
features_list = [row.features for row in rescaledData.select("features").collect()]
dense_features = np.array([v.toArray() for v in features_list])
similarity_matrix = cosine_similarity(dense_features)

print("--- 📊 코사인 유사도 매트릭스 ---")
print(similarity_matrix)

# 이제 0 / 1 문장은 0.15
# 2 / 3 문장은 0.11 이 되었다

--- 📝 토큰화 및 N-gram 결과 ---
+---+----------------------------------------+--------------------------------------------+--------------------------------------------------------------------+
|id |sentence                                |words                                       |ngrams                                                              |
+---+----------------------------------------+--------------------------------------------+--------------------------------------------------------------------+
|0  |아이브 해외투어는 성공적으로 끝난거 같다|[아이브, 해외, 투어, 성공, 적, 끝난거, 같다]|[아이브 해외, 해외 투어, 투어 성공, 성공 적, 적 끝난거, 끝난거 같다]|
|1  |이번 아이브 해외투어는 다소 아쉬웠다    |[이번, 아이브, 해외, 투어, 다소, 아쉬웠다]  |[이번 아이브, 아이브 해외, 해외 투어, 투어 다소, 다소 아쉬웠다]     |
|2  |이번에 이서는 특이한 의상을 입었다      |[이번, 이서, 특이한, 의상, 입었다]          |[이번 이서, 이서 특이한, 특이한 의상, 의상 입었다]                  |
|3  |이서도 특이한 노래도 냈었다             |[이서, 특이한, 노래, 냈었다]                |[이서 특이한, 특이한 노래, 노래 냈었다]                             |
+---+--------------------------------------